# 01 — Baseline and Tuning

Claude vs Gemini on three Level 1 subagents: what the baseline measured, and what
the ablation ladder bought.

**This notebook displays; it does not compute.** Every table and chart comes out of
`amw.reporting.notebook` over an artifact written by `cli.py`. If a cell here looks
like it is doing analysis, that is a bug — the analysis belongs in the library where
it can be tested.

Read the banner in the next output before reading any number below it.

In [ ]:
# papermill parameters
MODE = "replay"          # replay | hybrid | live
CUSTOMER = "demo_patents"
LIVE_CASES = 10          # size of the one live cell at the bottom

In [ ]:
%matplotlib inline
# the inline backend renders figures to PNG with no display attached, which is
# what papermill needs; `matplotlib.use("Agg")` would run headless too but would
# emit `<Figure ...>` text instead of a chart.

from IPython.display import Markdown, display

from amw.agents.schemas import SUBAGENTS
from amw.config import load_all
from amw.reporting import notebook as nb

cfg = load_all(customer=CUSTOMER)
phase2 = nb.load_phase2()
display(Markdown(f"**{nb.replay_banner(phase2)}**"))

## 1. Deterministic metrics

No judge involved: exact match, F1, and schema validity computed against the gold
references in the dataset. These are the numbers that do not depend on anyone's
opinion, which is why they come first.

In [ ]:
metrics = nb.metrics_frame(phase2)
display(metrics[nb.METRIC_COLUMNS])

### Schema validity, and the one thing you must say out loud about it

The `json_schema_validity` gate wants a CI lower bound at or above 0.99. Gemini
clears it. Claude does not — and the reason is a policy configuration, not the
model. The caveat below is printed from the same constant the scorecard renders,
so the two cannot drift apart.

In [ ]:
schema = metrics[metrics["metric"] == "json_schema_validity"]
fig = nb.interval_chart(
    schema,
    label=["subagent", "variant"],
    title="json_schema_validity — 95% CI, gate bound marked",
    bound=cfg.gates.subagent_gates["json_schema_validity"].min,
    bound_label="gate",
)
display(fig)

from amw.reporting import CLAUDE_SCHEMA_CAVEAT
display(Markdown(f"> **On every Claude figure above:** {CLAUDE_SCHEMA_CAVEAT}"))

## 2. Judge scores

Rubric-anchored, k=2 repeats, bootstrap CIs.

**`judged_n` and `split` are part of the number.** Feature Extractor was judged on
all 70 items; the other two on the 28-item core split. A judge column read without
those two columns compares two different measurements.

In [ ]:
judges = nb.judge_frame(phase2)
display(judges)

fig = nb.interval_chart(
    judges,
    label=["subagent", "variant"],
    title="Judge score — 95% CI (see judged_n above; splits differ)",
    xlabel="rubric-anchored judge score, 95% CI",
)
display(fig)

## 3. The ablation ladder

What did each prompt change actually buy? Each rung runs through the *same* scoring
path as the baseline, so the rungs are comparable to each other and to the baseline's
**core-split** judge numbers — not to the 70-item column above.

Rungs nobody has run stay in the table with `rung_status = no_recordings` and no
numbers. They are hypotheses, and a ladder that hid them would read as a ladder that
had been fully climbed.

In [ ]:
ladders = {s: nb.load_ablation(s) for s in SUBAGENTS}
for subagent, result in ladders.items():
    display(Markdown(f"**{subagent}**"))
    display(nb.ablation_frame(result)[nb.LADDER_COLUMNS])

In [ ]:
fe = nb.ablation_frame(ladders["feature_extractor"])
display(
    nb.interval_chart(
        fe,
        label=["rung", "output_mode"],
        title="Feature Extractor ladder — judge score, 95% CI (core split, n=28)",
    )
)

### What the ladder is telling us, and what it is not

Two things to read off the Feature Extractor table before drawing a conclusion:

- **A1–A3 is one bundled rung.** Prompt wording and output mode changed together, so
  the drop cannot be attributed to either alone. The `A0-schema` and
  `A4-novelty-*` rungs exist precisely to separate them, and they are unmeasured.
- **`leaked_example_items`** flags any rung whose worked example is itself a scored
  item. Where it is non-empty, that rung's number is optimistic by an unknown amount.

In [ ]:
for subagent, result in ladders.items():
    frame = nb.ablation_frame(result)
    leaked = frame[frame["leaked_example_items"] != ""]
    for _, row in leaked.iterrows():
        print(f"{subagent}/{row['rung']}: worked example is a scored item — {row['leaked_example_items']}")
    for note in result.notes:
        print(f"{subagent}: {note}")

## 4. One live cell

Everything above is replay. This cell is the live proof, parameterized by `MODE`:
in `replay` it resolves from recordings and makes zero calls; set `MODE="hybrid"` or
`"live"` and the same code path makes real ones. Same function `cli.py phase2` calls
— a demo that exercised a different path would prove nothing about the path a
customer will use.

In [ ]:
from amw.eval.runner import run_phase2

live = run_phase2(config=cfg, mode=MODE, n=LIVE_CASES, run_judge=False, write=False)
display(Markdown(f"**{nb.replay_banner(live)}**"))
display(nb.metrics_frame(live)[nb.METRIC_COLUMNS])
print(f"call errors: {sum(arm.calls_error for arm in live.arms)}")